# OpenPlaque — Secondary Branch 3-D Vesselness / Topology Experiment

This experiment replaces local 2-D connected-component continuation with a 3-D multiscale Hessian vesselness field and topology-aware graph search.

It starts **inside the known-good secondary branch (~10.6 mm)** and must first rediscover the known course to **~12.7 mm** as a positive control. Only after that succeeds is target-free distal continuation evaluated.

Research use only. No LCX identity is assigned automatically.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Reuse controls. Runtime -> Run all is supported.
# Existing valid caches are reused only when both the flag and cache are present.
REUSE_SOURCE_CT = True
REUSE_FROZEN_GEOMETRY = True
REUSE_VESSELNESS = True
REUSE_TOPOLOGY_SEARCH = True
REUSE_FIGURES = True
REUSE_REPORT = True


In [ ]:
!pip -q install scipy pandas matplotlib


In [ ]:
import os, shutil
if os.path.exists('/content/OpenPlaque'):
    shutil.rmtree('/content/OpenPlaque')
!git clone -q --depth 1 --branch secondary-3d-vesselness-topology-from-main https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%cd /content/OpenPlaque


In [ ]:
import sys
sys.path.insert(0, '/content/OpenPlaque/src')
from openplaque.secondary_3d_vesselness_topology import (
    Secondary3DVesselnessTopologyWorkflow,
    synthetic_vesselness_self_test,
)

self_test = synthetic_vesselness_self_test()
print('Synthetic 3-D vesselness self-test:', self_test)
assert self_test['passed'], 'Synthetic vesselness self-test failed'


In [ ]:
reuse = {
    'source_ct': REUSE_SOURCE_CT,
    'frozen_geometry': REUSE_FROZEN_GEOMETRY,
    'vesselness': REUSE_VESSELNESS,
    'topology_search': REUSE_TOPOLOGY_SEARCH,
    'figures': REUSE_FIGURES,
    'report': REUSE_REPORT,
}
wf = Secondary3DVesselnessTopologyWorkflow(reuse=reuse)
display(wf.cache_status())


In [ ]:
wf.load_source_ct()
geometry = wf.load_frozen_geometry(start_arc_mm=10.6, control_arc_mm=12.7)
display(geometry)


In [ ]:
vmeta = wf.compute_vesselness(scales_mm=(0.55, 0.80, 1.10, 1.45))
display(vmeta)


In [ ]:
summary = wf.search_topology(min_candidate_projection_mm=1.5, max_cost=150.0)
display(summary)
display(wf.candidates.head(12) if wf.candidates is not None and len(wf.candidates) else wf.candidates)


In [ ]:
wf.make_figures()
from IPython.display import display, Image
for name in [
    '01_vesselness_geometry.png',
    '02_positive_control.png',
    '03_distal_cross_sections.png',
    '04_path_profiles.png',
]:
    display(Image(filename=str(wf.out / name)))


In [ ]:
report = wf.make_report()
zip_path = wf.package()
print('Report:', report)
print('Final ZIP:', zip_path)

from urllib.parse import quote
final_name = 'OPENPLAQUE_SECONDARY_3D_VESSELNESS_TOPOLOGY_REPORT_BACK.zip'
drive_search = 'https://drive.google.com/drive/u/0/search?q=' + quote(final_name)
print('Drive search link for final result:')
print(drive_search)
